In [ ]:
pip install pdfminer.six

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 32.3 MB/s eta 0:00:00


In [ ]:
import csv
from pdfminer.high_level import extract_pages
from pdfminer.layout import LTTextContainer, LTChar

output = []

for page_layout in extract_pages("The Flavor Bible.pdf"):
    for element in page_layout:
        if isinstance(element, LTTextContainer):
            for text_line in element:
                font_sizes = []
                line_text = text_line.get_text().strip()
                for character in text_line:
                    if isinstance(character, LTChar):
                        font_sizes.append(round(character.size, 1))
                if font_sizes and line_text:
                    avg_font = sum(font_sizes) / len(font_sizes)
                    output.append((avg_font, line_text))

# Save to CSV
with open("flavor_bible_fontsize.csv", "w", newline='', encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["Font Size", "Text"])
    writer.writerows(output)


In [ ]:
import csv
import json

csv_file_path = "flavor_bible_fontsize.csv"
output = []

with open(csv_file_path, newline='', encoding='utf-8') as csvfile:
    reader = csv.reader(csvfile)
    next(reader)  # skip header
    rows = []

    for row in reader:
        if len(row) == 2:
            try:
                size = float(row[0])
                text = row[1]
                rows.append((size, text))
            except ValueError:
                continue  # skip rows with non-numeric font sizes

tolerance = 0.01  # This can be adjusted based on how precise you want the comparison to be

# Process ingredients based strictly on font size with tolerance
for i, (font_size, ingredient_line) in enumerate(rows):
    if abs(font_size - 16.6) <= tolerance:  # Allowing a small tolerance
        if i + 2 < len(rows):
            _, taste_line = rows[i + 2]
            if taste_line.lower().startswith("taste:"):
                output.append({
                    "ingredient": ingredient_line,
                    "taste": taste_line
                })


# Save to JSON
with open("all_ingredients_with_exact_taste_lines.json", "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2)

print("✅ Done. File saved as all_ingredients_with_exact_taste_lines.json")


✅ Done. File saved as all_ingredients_with_exact_taste_lines.json


In [ ]:
import json

# Function to clean taste string by removing unwanted characters and spaces
def clean_taste(taste_string):
    # Replace "Taste:" part and remove tabs and special characters
    cleaned_taste = taste_string.replace("Taste:", "").replace("\t", "").strip()

    # Handle specific unicode characters
    cleaned_taste = cleaned_taste.replace("\u2013", "-")  # Replace en dash with normal dash
    cleaned_taste = cleaned_taste.replace("\u2014", "-")  # Replace em dash with normal dash
    cleaned_taste = cleaned_taste.replace("\u2018", "'")  # Replace left single quote with regular quote
    cleaned_taste = cleaned_taste.replace("\u2019", "'")  # Replace right single quote with regular quote
    cleaned_taste = cleaned_taste.replace("\u201C", '"')  # Replace left double quote with regular quote
    cleaned_taste = cleaned_taste.replace("\u201D", '"')  # Replace right double quote with regular quote
    cleaned_taste = cleaned_taste.replace("\xa0", " ")  # Replace non-breaking space with regular space

    return cleaned_taste

# Function to clean ingredient name (remove tabs and special characters)
def clean_ingredient(ingredient_string):
    ingredient_string = ingredient_string.replace("\t", " ").strip()

    # Handle specific unicode characters in ingredient names
    ingredient_string = ingredient_string.replace("\u2013", "-")  # Replace en dash with normal dash
    ingredient_string = ingredient_string.replace("\u2014", "-")  # Replace em dash with normal dash
    ingredient_string = ingredient_string.replace("\xa0", " ")  # Replace non-breaking space with regular space

    return ingredient_string

# Load the obtained taste profile JSON file (with ingredient and taste)
with open('Raw_flavor_profile.json', 'r', encoding='utf-8') as f:
    taste_profiles_data = json.load(f)

# List to store cleaned ingredients with taste profiles
cleaned_data = []

# Process each taste profile
for profile in taste_profiles_data:
    ingredient = clean_ingredient(profile['ingredient'])
    taste = clean_taste(profile['taste'])

    # Only add to the final list if the taste profile is not empty
    if taste:
        cleaned_data.append({
            "ingredient": ingredient,
            "taste": taste
        })

# Save the cleaned data into a new JSON file
with open('cleaned_ingredients_with_taste_profiles.json', 'w', encoding='utf-8') as f:
    json.dump(cleaned_data, f, indent=2)

print("✅ Cleaned data saved to 'cleaned_ingredients_with_taste_profiles.json'.")


✅ Cleaned data saved to 'cleaned_ingredients_with_taste_profiles.json'.


In [ ]:
import json

# Function to compare and find the taste profile
def find_taste_profile(old_ingredient, new_data):
    for item in new_data:
        # If the old ingredient is found within the new ingredient (case insensitive)
        if old_ingredient.lower() in item['ingredient'].lower():
            return item['taste']
    return "no profile found"

# Load the old JSON file (your ingredients with no taste profile)
with open('ingredient_taste_profiles.json', 'r', encoding='utf-8') as f:
    old_ingredients = json.load(f)

# Load the new JSON file (the one with cleaned ingredients and taste profiles)
with open('cleaned_ingredients_with_taste_profiles.json', 'r', encoding='utf-8') as f:
    cleaned_ingredients = json.load(f)

# List to store the results
result = []

# Compare ingredients and find the corresponding taste profiles
for old_item in old_ingredients:
    old_ingredient = old_item['Ingredient']  # Assuming the old JSON has "Ingredient" field

    # Find the corresponding taste profile
    taste_profile = find_taste_profile(old_ingredient, cleaned_ingredients)

    # Append the result
    result.append({
        "ingredient": old_ingredient,
        "taste profile": taste_profile
    })

# Save the result to a new JSON file
with open('ingredients_with_taste_profiles_comparison.json', 'w', encoding='utf-8') as f:
    json.dump(result, f, indent=2)

print("✅ Comparison completed. File saved as 'ingredients_with_taste_profiles_comparison.json'.")


✅ Comparison completed. File saved as 'ingredients_with_taste_profiles_comparison.json'.


In [ ]:
import json

# Load the obtained comparison JSON file
with open('IngredientFlavourProfile.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# Display first few entries to understand the structure
for item in data[:5]:
    print(item)

{'ingredient': 'Salt', 'taste profile': 'no profile found'}
{'ingredient': 'Turmeric', 'taste profile': 'bittersweet;pungent'}
{'ingredient': 'Cumin', 'taste profile': 'no profile found'}
{'ingredient': 'Coriander', 'taste profile': 'no profile found'}
{'ingredient': 'Sugar', 'taste profile': 'sweet'}


In [ ]:
# Count ingredients with "no profile found"
missing_profile_count = sum(1 for item in data if item['taste profile'] == "no profile found")
print(f"Number of ingredients without a matching profile: {missing_profile_count}")


Number of ingredients without a matching profile: 199


In [ ]:
from collections import Counter

# Extract all the taste profiles
taste_profiles = [item['taste profile'] for item in data]

# Count the occurrences of each taste profile
taste_profile_counts = Counter(taste_profiles)
print(f"Taste profile distribution: {taste_profile_counts}")


Taste profile distribution: Counter({'no profile found': 199, 'sweet': 28, 'bitter': 7, 'sour': 4, 'sour,sweet': 3, 'sweet,astringent': 3, 'bitter,sweet': 2, 'sour,hot': 2, 'astringent': 2, 'bittersweet;pungent': 1, 'pungent(+sweetwithcookingviacaramelization)': 1, 'sweet,sour': 1, 'sweet-stringent': 1, 'sweet,bitter,pungent': 1, 'bittertosweet,fromunripe(green)toripe(yellowtored)': 1, 'pungent': 1, 'pungent,hot': 1})
